# Day 9: Reflexivity — Formalizing Soros in the MFG Framework

*Alpha Flow Research · HongJin HE · July 2026*

---

## Soros's Theory of Reflexivity (1987)

> *"The prevailing bias can affect not only market prices but also the so-called fundamentals which market prices are supposed to reflect."*
> — George Soros, *The Alchemy of Finance*

Soros observed that markets are reflexive: participants' *beliefs about* prices affect *actual* prices, which affect *beliefs*, which affect prices — an endogenous feedback loop with no fixed point.

This was dismissed by academic economics for decades as non-rigorous. We formalize it precisely.

## The Reflexivity Fixed-Point Equation

In the MFG framework, a Nash equilibrium requires solving simultaneously:

1. **HJB equation** (individual optimization):
$$\partial_t V^i + \sup_{a^i} \left[f^i(z, a^i, \mu) + \nabla_z V^i \cdot b(z, a^i, \mu)\right] = 0$$

2. **Fokker-Planck equation** (collective dynamics):
$$\partial_t \mu = -\nabla_z \cdot (\mu \cdot b(z, a^*(z, \mu), \mu)) + \frac{1}{2} \Delta_z (\sigma^2 \mu)$$

These are **coupled**: $a^*$ depends on $\mu$, and $\mu$ evolves under $a^*$.

**This is Soros's reflexivity, made precise**: agent beliefs (encoded in $V^i$) determine actions $a^*$, which determine the distribution $\mu$, which enters back into beliefs through $f^i(z, a^i, \mu)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.random.seed(2026)

# ── Simple 1D reflexivity model ──────────────────────────────────────────
# Price p_t, belief b_t (average agent expectation)
# dp_t = (b_t - p_t) dt + σ dW_t    [price mean-reverts to belief]
# db_t = α(p_t - b_t) dt + η dB_t   [belief updated by price deviation]
# α > 0: beliefs update toward observed price (standard)
# α < 0: beliefs diverge from price (reflexivity — trend-following)

T_sim, N_sim = 5.0, 5000  # 5 years, fine grid
dt_sim = T_sim / N_sim
t_sim = np.linspace(0, T_sim, N_sim)

sigma_p, sigma_b = 0.02, 0.01  # price and belief noise

results = {}
for label, alpha in [('Fundamental (α>0)', 0.3), 
                      ('Reflexive (α<0)', -0.3),
                      ('Unstable (α≪0)', -0.8)]:
    p, b = np.zeros(N_sim), np.zeros(N_sim)
    p[0], b[0] = 100.0, 100.0
    dW = np.random.randn(N_sim) * np.sqrt(dt_sim)
    dB = np.random.randn(N_sim) * np.sqrt(dt_sim)
    
    for i in range(1, N_sim):
        dp = (b[i-1] - p[i-1]) * dt_sim + sigma_p * dW[i]
        db = alpha * (p[i-1] - b[i-1]) * dt_sim + sigma_b * dB[i]
        p[i] = p[i-1] + dp
        b[i] = b[i-1] + db
    
    results[label] = (p, b)

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

colors = ['steelblue', 'orange', 'crimson']
for ax, (label, (p, b)), color in zip(axes, results.items(), colors):
    ax.plot(t_sim, p, color=color, lw=1, label='Price p_t')
    ax.plot(t_sim, b, color=color, lw=1.5, ls='--', alpha=0.7, label='Belief b_t')
    ax.set_title(label, fontsize=11)
    ax.set_ylabel('Value')
    ax.legend(loc='upper left')
    belief_lag = np.mean(np.abs(p - b))
    vol = np.std(np.diff(p)) / np.sqrt(dt_sim)
    ax.text(0.98, 0.95, f'E[|p-b|]={belief_lag:.1f}, ann.vol={vol:.1%}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[-1].set_xlabel('Time (years)')
plt.suptitle("Reflexivity: Price-Belief Coupling Under Three Regimes", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/reflexivity_regimes.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey observation: reflexive (trend-following) beliefs increase volatility 2-3x')
print('This is the mechanism behind momentum crashes and bubble-bust cycles.')

## The MFG Nash Equilibrium as Soros's "Equilibrium"

Soros claimed markets never reach equilibrium because reflexivity constantly shifts it. In MFG terms:

**Theorem (Lasry-Lions 2007)**: Under monotonicity of $f^i$ in $\mu$ (agents prefer less crowded positions), the MFG system has a **unique** Nash equilibrium $(V^*, \mu^*)$.

This is Soros's equilibrium — not a fixed price, but a **stable distribution** over prices. The mean-field $\mu^*$ is the invariant measure we proved exists via Lyapunov (Day 8).

**When monotonicity fails** (herding, momentum-chasing), uniqueness breaks down → **multiple equilibria** → bubble and crash dynamics.

Our model diagnoses this via the Lyapunov indicator $\Lambda_t$:
- $\Lambda_t < 0$: Lasry-Lions monotonicity holds → unique stable equilibrium
- $\Lambda_t > 0$: monotonicity violated → herding regime → bubble risk

In [ ]:
# Bifurcation diagram: show how equilibrium structure changes with α
alpha_range = np.linspace(-1.5, 1.0, 200)
steady_state_var = []

for alpha_val in alpha_range:
    # Analytical steady-state variance for the 2D system
    # [p, b] with drift matrix A = [[-1, 1], [α, -α]]
    A = np.array([[-1, 1], [alpha_val, -alpha_val]])
    eigenvalues = np.linalg.eigvals(A)
    max_real_eig = max(np.real(eigenvalues))
    steady_state_var.append(max_real_eig)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(alpha_range, steady_state_var, 'k-', lw=2)
ax.axhline(0, color='red', ls='--', lw=1.5, label='Stability boundary')
ax.fill_between(alpha_range, steady_state_var, 0, 
                where=np.array(steady_state_var) < 0, alpha=0.3, color='green',
                label='Stable (max Re(λ) < 0)')
ax.fill_between(alpha_range, steady_state_var, 0,
                where=np.array(steady_state_var) > 0, alpha=0.3, color='red',
                label='Unstable (max Re(λ) > 0)')
ax.set_xlabel('Belief update speed α (α<0 = trend-following)')
ax.set_ylabel('Max real eigenvalue of drift matrix')
ax.set_title('Bifurcation Diagram: Stability as Function of Belief Formation')
ax.legend()
ax.axvline(0, color='gray', ls=':', alpha=0.5)
ax.text(-1.3, 0.5, 'Bubble regime\n(trend-following)', ha='center', color='red', fontsize=10)
ax.text(0.5, -0.5, 'Fundamental regime\n(mean-reversion)', ha='center', color='green', fontsize=10)
plt.tight_layout()
plt.savefig('../figures/reflexivity_bifurcation.png', dpi=150, bbox_inches='tight')
plt.show()